# Deep Dive: Evaluating Agents & Computer Use

## 📋 Summary

Shipping an agent without automated evaluation is flying blind — and automating a GUI without strict sandboxing and coordinate scaling is asking for misclicks or security incidents.

In this final notebook (covering **Lessons 12–13**: Evaluating Agents and Computer Use), you will:
1. Build a **Trace-Based Evaluation Harness** for a **Recipe Nutrition Agent** with three orthogonal evaluators and **partial-credit scoring** (`0.0`–`1.0`):
   - **Tool Call Evaluator** (stepped partial credit: wrong tool vs missing params vs wrong values)
   - **Schema Validity Evaluator** (Pydantic validation + semantic checks)
   - **Accuracy Evaluator** (numeric tolerance + optional LLM-as-judge)
2. Build a **Safe Computer Use Controller** (mock GUI environment) implementing coordinate scaling (`<= 1568px` / `<= 1.15M px`), the screenshot-action feedback loop, action caps, audit logs, and confirmation gates.

### Key concepts in this notebook

| # | Concept | Complexity |
|---|---|---|
| 1 | Separating agent execution (`AgentTrace`) from evaluation | 🟢 Beginner |
| 2 | Multi-dimensional evaluators with partial credit (`0.0`–`1.0`) | 🟡 Intermediate |
| 3 | Aggregate evaluation reporting & regression tracking | 🟡 Intermediate |
| 4 | Computer Use: display resolution & coordinate scaling math | 🟡 Intermediate |
| 5 | The screenshot-action loop & defense-in-depth safety guardrails | 🔴 Advanced |
| 6 | Hands-on evaluator & computer-use safety exercises | 🏋️ Practice |

> **How to use this notebook:** Sections 1–4 run both offline and with live Anthropic API calls. Section 5 runs safely in **mock mode** (no real OS clicks) so you can experiment without risk.

## 0. Setup

In [ ]:
%pip install -q anthropic pydantic

import json
import math
import os
from dataclasses import dataclass, field
from getpass import getpass
from typing import Any
from pydantic import BaseModel, Field, ValidationError

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    if not os.environ.get("ANTHROPIC_API_KEY"):
        os.environ["ANTHROPIC_API_KEY"] = getpass("Enter your ANTHROPIC_API_KEY: ")

import anthropic

client = anthropic.Anthropic()
MODEL = "claude-haiku-4-5-20251001"
print("Setup complete. Model:", MODEL)

## 1. Capturing `AgentTrace` for a Recipe Nutrition Agent 🟢

In Lesson 12, agents do **not** grade themselves. Instead, the agent runs and returns a structured `AgentTrace` containing:
- `tool_calls`: every tool name + input dictionary invoked during the run
- `result`: the final structured output (or `None` if parsing failed)

Separating **trace capture** from **grading** lets you re-run new evaluators over historical traces without paying for new LLM calls.

In [ ]:
class NutritionSummary(BaseModel):
    dish_name: str = Field(..., min_length=1)
    total_calories: float = Field(..., gt=0)
    protein_g: float = Field(..., ge=0)
    is_high_protein: bool


@dataclass
class AgentTrace:
    tool_calls: list[dict[str, Any]] = field(default_factory=list)
    result: dict[str, Any] | None = None


@dataclass
class EvalCase:
    name: str
    prompt: str
    expected_tool: str
    expected_servings: int
    expected_calories: float
    expected_high_protein: bool


# Deterministic business logic (kept separate from tool wrapper, as in Lesson 12)
NUTRITION_DB = {
    "salmon_bowl": {"cal_per_serving": 520.0, "protein_per_serving": 38.0},
    "pasta_marinara": {"cal_per_serving": 410.0, "protein_per_serving": 12.0},
}


def compute_nutrition(dish_key: str, servings: int) -> dict:
    item = NUTRITION_DB[dish_key]
    total_cal = round(item["cal_per_serving"] * servings, 2)
    protein = round(item["protein_per_serving"] * servings, 2)
    return {"dish_key": dish_key, "servings": servings, "total_calories": total_cal, "protein_g": protein}

## 2. Three Orthogonal Evaluators with Partial Credit 🟡

Mirroring Lesson 12's evaluator contract (`{name, passed, score, details}`):
1. **`eval_tool_call`**: `0.0` = no tool call; `0.25` = wrong tool; `0.50` = missing required params; `0.75` = wrong param values; `1.0` = exact tool & params.
2. **`eval_schema`**: `0.0` = missing result; `0.5` = Pydantic validation failed; `1.0` = valid schema.
3. **`eval_accuracy`**: checks `total_calories` within tolerance (`<= 1.0` kcal) and `is_high_protein` boolean match (`0.5` each).

In [ ]:
@dataclass
class EvaluatorResult:
    name: str
    passed: bool
    score: float
    details: str


def eval_tool_call(trace: AgentTrace, case: EvalCase) -> EvaluatorResult:
    if not trace.tool_calls:
        return EvaluatorResult("tool_call", False, 0.0, "No tool was called")
    call = trace.tool_calls[0]
    if call["name"] != case.expected_tool:
        return EvaluatorResult("tool_call", False, 0.25, f"Wrong tool: {call['name']}")
    inp = call.get("input", {})
    if "dish_key" not in inp or "servings" not in inp:
        return EvaluatorResult("tool_call", False, 0.50, f"Missing params: {inp}")
    if inp["servings"] != case.expected_servings:
        return EvaluatorResult("tool_call", False, 0.75, f"Expected servings={case.expected_servings}, got {inp['servings']}")
    return EvaluatorResult("tool_call", True, 1.0, "Exact tool and parameters")


def eval_schema(trace: AgentTrace, _case: EvalCase) -> EvaluatorResult:
    if trace.result is None:
        return EvaluatorResult("schema", False, 0.0, "No structured result returned")
    try:
        NutritionSummary.model_validate(trace.result)
        return EvaluatorResult("schema", True, 1.0, "Valid NutritionSummary schema")
    except ValidationError as e:
        return EvaluatorResult("schema", False, 0.5, f"Schema error: {e.error_count()} issue(s)")


def eval_accuracy(trace: AgentTrace, case: EvalCase, tol: float = 1.0) -> EvaluatorResult:
    if not trace.result:
        return EvaluatorResult("accuracy", False, 0.0, "No result to grade")
    cal_ok = abs(trace.result.get("total_calories", -1) - case.expected_calories) <= tol
    hp_ok = trace.result.get("is_high_protein") == case.expected_high_protein
    score = (0.5 if cal_ok else 0.0) + (0.5 if hp_ok else 0.0)
    return EvaluatorResult("accuracy", score == 1.0, score, f"cal_ok={cal_ok}, high_protein_ok={hp_ok}")


def run_eval_suite(cases_and_traces: list[tuple[EvalCase, AgentTrace]]) -> None:
    scores = []
    for case, trace in cases_and_traces:
        results = [eval_tool_call(trace, case), eval_schema(trace, case), eval_accuracy(trace, case)]
        avg = sum(r.score for r in results) / len(results)
        passed = all(r.passed for r in results)
        scores.append(avg)
        status = "PASS" if passed else "FAIL"
        print(f"[{status}] {case.name:22s} | avg={avg:.2f} | " + " | ".join(f"{r.name}={r.score:.2f}" for r in results))
    print(f"\nSuite Average Score: {sum(scores)/len(scores)*100:.1f}%")


# Demonstrate full pass vs partial credit failure diagnosis
cases = [
    (
        EvalCase("salmon_2x", "2 servings salmon_bowl", "mcp__nutrition__compute", 2, 1040.0, True),
        AgentTrace(
            tool_calls=[{"name": "mcp__nutrition__compute", "input": {"dish_key": "salmon_bowl", "servings": 2}}],
            result={"dish_name": "Salmon Bowl", "total_calories": 1040.0, "protein_g": 76.0, "is_high_protein": True},
        ),
    ),
    (
        EvalCase("pasta_wrong_servings", "3 servings pasta_marinara", "mcp__nutrition__compute", 3, 1230.0, False),
        AgentTrace(
            tool_calls=[{"name": "mcp__nutrition__compute", "input": {"dish_key": "pasta_marinara", "servings": 2}}],
            result={"dish_name": "Pasta Marinara", "total_calories": 820.0, "protein_g": 24.0, "is_high_protein": False},
        ),
    ),
]
run_eval_suite(cases)

### 🧐 Observation
- Look at `pasta_wrong_servings`: binary pass/fail would just say `FAIL`. Partial credit immediately pinpoints **why**: `schema=1.00`, `tool_call=0.75` (right tool, wrong `servings`), `accuracy=0.50` (calories off because of wrong servings).

## 3. Computer Use: Display Resolution & Coordinate Scaling 🟡

Lesson 13 introduces Claude's **Computer Use** beta (`computer_20250124`, beta header `computer-use-2025-01-24`). Because high-DPI displays (e.g., 2560x1440) exceed Claude's optimal vision bounds (**longest edge `<= 1568px`** and **total pixels `<= 1,150,000`**):
1. Downscale screenshots by `scale_factor` before sending to Claude.
2. When Claude returns click coordinates `[x, y]` in the downscaled image, **divide by `scale_factor`** to map back to native display pixels!

In [ ]:
def get_scale_factor(width: int, height: int) -> float:
    return min(
        1.0,
        1568 / max(width, height),
        math.sqrt(1_150_000 / (width * height)),
    )


def to_native_coords(x: int, y: int, scale_factor: float) -> tuple[int, int]:
    return (round(x / scale_factor), round(y / scale_factor))


for w, h in [(1024, 768), (1920, 1080), (2560, 1440)]:
    sf = get_scale_factor(w, h)
    scaled_w, scaled_h = round(w * sf), round(h * sf)
    native = to_native_coords(500, 300, sf)
    print(f"Native {w}x{h} -> scale={sf:.3f} -> model sees {scaled_w}x{scaled_h} | model(500,300) -> native{native}")

## 4. Safe Mock Computer-Use Loop & Defense-in-Depth 🔴

Real computer use must **always** run inside an isolated sandbox (Docker/VM) with:
- Domain allowlist / blocklist
- Hard `max_actions` cap
- Confirmation gate for sensitive actions (`submit`, `payment`, `login`, `delete`)
- Full action audit log

Below is a safe mock controller simulating a **hotel booking form** automation loop without touching your real screen.

In [ ]:
@dataclass
class SafetyConfig:
    sandbox_mode: str = "mock"  # "mock", "docker", or "vm"
    max_actions: int = 8
    allowed_domains: tuple[str, ...] = ("booking.sandbox.local",)
    confirm_keywords: tuple[str, ...] = ("payment", "delete", "login", "submit_charge")


class SafeMockComputerController:
    def __init__(self, safety: SafetyConfig):
        if safety.sandbox_mode not in ("mock", "docker", "vm"):
            raise RuntimeError("Refusing to run Computer Use outside a sandbox!")
        self.safety = safety
        self.action_log: list[dict[str, Any]] = []

    def execute(self, action: str, **params: Any) -> dict[str, Any]:
        if len(self.action_log) >= self.safety.max_actions:
            return {"success": False, "error": f"Action cap reached ({self.safety.max_actions})"}

        # Gate sensitive actions
        payload_str = json.dumps(params).lower()
        if any(kw in action.lower() or kw in payload_str for kw in self.safety.confirm_keywords):
            result = {"success": False, "requires_confirmation": True, "error": f"Blocked sensitive action: {action}"}
        else:
            result = {"success": True, "action": action, "params": params, "mock_screenshot": "<base64-png>"}

        self.action_log.append({"step": len(self.action_log) + 1, "action": action, "params": params, "result": result})
        return result


ctrl = SafeMockComputerController(SafetyConfig(max_actions=5))
for act, kwargs in [
    ("screenshot", {}),
    ("left_click", {"coordinate": [320, 210]}),
    ("type", {"text": "2026-07-10"}),
    ("left_click", {"coordinate": [420, 520], "label": "payment_confirm"}),
]:
    res = ctrl.execute(act, **kwargs)
    print(f"{act:12s} -> success={res['success']} | {res.get('error', 'OK')}")

### Failure Modes & Fixes

| Failure Mode | Why It Happens | Fix |
|---|---|---|
| Binary evals hide root cause | Only checking final string equality | Use orthogonal evaluators (tool call, schema, accuracy) with `0..1` partial credit |
| Flaky numeric evals | Strict `==` on floating-point outputs | Use explicit tolerance (`abs(a - b) <= tol`) |
| Misclicks on HiDPI screens | Passing raw 2560x1440 screenshots without scaling back | Compute `get_scale_factor` and divide returned coords by `scale_factor` |
| Unintended clicks / prompt injection | Running Computer Use on host OS without gates | Require Docker/VM sandbox, domain allowlists, `max_actions`, and human confirmation on sensitive actions |

## 5. Exercises 🏋️

### Exercise 1: Add a Live Trace-Capturing Nutrition Agent
Write `run_nutrition_agent(prompt: str) -> AgentTrace` that gives Claude a `mcp__nutrition__compute` tool, runs the tool loop, extracts a `NutritionSummary`, and returns an `AgentTrace`. Then evaluate it with `run_eval_suite(...)`.

In [ ]:
def run_nutrition_agent(prompt: str) -> AgentTrace:
    """Call Claude with the nutrition compute tool, capture tool_calls and final NutritionSummary."""
    trace = AgentTrace()
    # TODO: Define the `mcp__nutrition__compute` tool + `submit_summary` tool,
    # run the Messages API loop, record tool_calls in `trace.tool_calls`,
    # and store the final dict in `trace.result`.
    return trace


# Test once implemented:
# live_case = EvalCase("live_salmon", "Calculate nutrition for 2 servings of salmon_bowl (high protein if >= 30g).", "mcp__nutrition__compute", 2, 1040.0, True)
# run_eval_suite([(live_case, run_nutrition_agent(live_case.prompt))])
print("Implement run_nutrition_agent above to run live evaluation.")

### Exercise 2: Enforce a Domain Allowlist in `SafeMockComputerController`
Add a `navigate(url: str)` check inside `SafeMockComputerController.execute` so that navigating to any hostname not in `self.safety.allowed_domains` is blocked and logged. Test with `"https://booking.sandbox.local/checkout"` (allowed) and `"https://evil.example.com/phish"` (blocked).

In [ ]:
from urllib.parse import urlparse


class DomainGuardedController(SafeMockComputerController):
    def execute(self, action: str, **params: Any) -> dict[str, Any]:
        # TODO: If action == "navigate", parse params["url"] with urlparse and block
        # if parsed.hostname not in self.safety.allowed_domains
        return super().execute(action, **params)


guarded = DomainGuardedController(SafetyConfig())
print(guarded.execute("navigate", url="https://booking.sandbox.local/rooms"))
print(guarded.execute("navigate", url="https://evil.example.com/phish"))

## 🏁 Summary & Key Takeaways

- **Evaluate traces, not just text:** Capture `tool_calls` and structured `result` in `AgentTrace`, then grade tool usage, schema validity, and accuracy with continuous `0..1` partial credit.
- **Scale coordinates for Computer Use:** Clamp vision inputs to `<= 1568px` / `<= 1.15M` pixels and divide returned coordinates by `scale_factor`.
- **Sandbox and gate GUI agents:** Never run Computer Use without Docker/VM isolation, domain allowlists, `max_actions` caps, audit logs, and confirmation gates for sensitive actions.

### What's next?
All 7 study notebooks for **01-ai_engineering** are complete! Next, open **`agent/agent-basic-knowledge.md`** to review the distilled knowledge agent for your upcoming project.